# Lecture Demo: Testing Relationships in Our Survey

**CP201A Planning Methods Gateway, Fall 2026 · Wednesday, November 18**

This is the notebook you'll use to run the statistical tests for **P/NP #11 (Part 2)** and for the survey findings in your **final memo**. It has three tests: a **t-test** to compare two group averages, a **Z-test of proportions** to compare two group shares, and a **chi-square test** for a crosstab.

### How this notebook fits with Lab 12

| Step | Where | What you get |
|---|---|---|
| 1. Clean the data, make your dummies, describe your two hypotheses | **Lab 12** notebook (this week's lab) | `survey_cleaned_2026.csv`, saved in your Lab 12 folder. Your Lab 12 notebook is **P/NP #11, Part 1**. |
| 2. Test your two hypotheses, make a chart, draft your sentences | **This notebook, Section 8** | The numbers, chart, and sentences for the **P/NP #11 form (Part 2)** and your final memo |
| 3. Correlations (two number variables) | Monday, November 23 notebook | One more test option |

**Do Lab 12 first.** Section 1 below looks for the cleaned file you saved in Lab 12 and loads it, so every dummy variable you made there is available here. (If it can't find that file, it cleans the raw class data the same way Lab 12 does, so the examples still run.)

**If you missed class:** Sections 3 to 7 are the examples from lecture, with the slides on bCourses. Section 8 is where you do your own tests, step by step.

**What you submit:** nothing from this notebook. You copy its results into the P/NP #11 form.

## 1. Setup: load the data

In [ ]:
import os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import chi2_contingency
from statsmodels.stats.proportion import proportions_ztest

pd.set_option('display.float_format', '{:.2f}'.format)

Run the next cell. It looks for **your** cleaned file from Lab 12 (`survey_cleaned_2026.csv`), first in this folder and then in your Lab 12 folder. If it finds it, you're set. If not, it cleans the raw class file the same way Lab 12 does, using the same variable names, so the lecture examples still run.

> **Use your own cleaned file for your own tests.** If the message below says it cleaned the raw file, you're looking at the *demo's* cleaning choices (which answers count as "agree," what became NaN, which dummies exist), not yours. That's fine for following along with the lecture. For P/NP #11 and your final memo, finish Lab 12, save `survey_cleaned_2026.csv`, and rerun this cell until it says **"Loaded YOUR cleaned Lab 12 file."**

In [ ]:
def clean_raw_survey(raw):
    '''The Lab 12 cleaning steps, condensed. See Lab 12 for what each step does and why.'''
    d = raw[raw['Unnamed: 0'] != 'Example'].drop(columns=['Unnamed: 0'])
    d = d[d['City'].notna()].reset_index(drop=True)
    d = d.map(lambda v: v.strip() if isinstance(v, str) else v)
    d['survey_id'] = d['Surveyor'] + '_' + d['Survey Number'].astype(str)
    d = d.rename(columns={
        'Interviewer or self-administered': 'mode', 'City': 'city', 'Survey Language': 'language',
        'What best describes your role here?': 'role',
        'About how many years have you worked or volunteered here?': 'years_worked',
        'Does this business or organization own or rent its space?': 'space_tenure',
        'Do you live in this neighborhood?': 'lives_here', 'Do you own or rent your home?': 'home_tenure',
        'On a typical workday, about how many minutes does it take you to get here?': 'commute_min',
        'This neighborhood has changed a lot in the past five years.': 'changed_a_lot',
        'Long-time residents can afford to stay in this neighborhood.': 'afford_to_stay',
        'I support building more housing in this neighborhood.': 'support_housing',
        'I feel connected to the people and community in this neighborhood.': 'connected',
        'Your own rent or mortgage payment': 'p12_own_housing'})

    def looks_like_date(x):
        return isinstance(x, str) and (re.match(r'^\d{4}-\d{2}-\d{2}', x) is not None
                                       or re.match(r'^\d{1,2}/\d{1,2}/\d{2,4}', x) is not None)
    def to_number(x, minutes=False):
        if pd.isna(x): return np.nan
        if isinstance(x, (int, float)): return float(x)
        s = str(x).lower()
        if looks_like_date(s) or 'know' in s or 'skip' in s: return np.nan
        m = re.match(r'^\s*(\d+(?:\.\d+)?)\s*-\s*(\d+(?:\.\d+)?)', s)
        if m: v = (float(m.group(1)) + float(m.group(2))) / 2
        else:
            nums = re.findall(r'\d+(?:\.\d+)?', s)
            if not nums: return np.nan
            v = float(nums[0])
        if minutes and ('hour' in s or 'hr' in s): v *= 60
        if not minutes and 'month' in s and 'year' not in s: v /= 12
        return v

    d['commute_num'] = d['commute_min'].apply(lambda x: to_number(x, minutes=True))
    d['years_worked_num'] = d['years_worked'].apply(to_number)
    agree_dummy = {'Strongly agree': 1, 'Agree': 1, 'Neutral': 0, 'Disagree': 0, 'Strongly disagree': 0}
    agree_score = {'Strongly disagree': 1, 'Disagree': 2, 'Neutral': 3, 'Agree': 4, 'Strongly agree': 5}
    for col in ['changed_a_lot', 'afford_to_stay', 'support_housing', 'connected']:
        d[col + '_agree_dv'] = d[col].map(agree_dummy)      # "Don't know" and blanks become NaN
        d[col + '_score'] = d[col].map(agree_score)
    d['p12_own_housing_increased_dv'] = d['p12_own_housing'].map(
        {'Increased a lot': 1, 'Increased a little': 1, 'Stayed about the same': 0,
         'Decreased a little': 0, 'Decreased a lot': 0})
    d['lives_here_dv'] = d['lives_here'].map({'Yes': 1, 'No': 0})
    d['rent_home_dv'] = d['home_tenure'].map({'Rent': 1, 'Own': 0})
    d['self_admin_dv'] = d['mode'].map({'Self-administered': 1, 'Interviewer': 0})
    return d

df = None
for path in ['survey_cleaned_2026.csv', '../Week13_Lab_CleaningSurveyData/survey_cleaned_2026.csv']:
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f'Loaded YOUR cleaned Lab 12 file: {path} ({len(df)} surveys).')
        break
if df is None:
    for path in ['CP201A_2026_survey_data.csv', '../Week13_Lab_CleaningSurveyData/CP201A_2026_survey_data.csv']:
        if os.path.exists(path):
            df = clean_raw_survey(pd.read_csv(path))
            print(f'No Lab 12 file found, so we cleaned the raw class file the demo way: {len(df)} surveys.')
            print("NOTE: these are the DEMO's cleaning choices, not yours. Fine for following along with the lecture.")
            print('For P/NP #11 and your final memo, finish Lab 12, save survey_cleaned_2026.csv, and rerun this cell.')
            break
if df is None:
    print('Could not find the data. Upload CP201A_2026_survey_data.csv (from bCourses) to this folder and rerun.')

## 2. The test functions (run this cell; you don't need to change it)

These functions do the work. Each one takes your data (`df`), your outcome (`y`), and your explanatory variable (`x`), and prints the number of people in each group, the shares or means, the test result, whether it's significant at your confidence level, and a **draft sentence** you can edit for your memo. Two helpers draw a bar chart with margins of error and make a "who responded" table for your methods section.

**Set your confidence level once, before you look at any results** (as in Assignment 1). Change `CONFIDENCE` in the first line if you want 90 or 99 percent.

*About the t-test:* `t_test_means` uses **Welch's t-test** (`equal_var=False`). Last year's notebook used the standard ("Student's") t-test, which assumes both groups are equally spread out. In our survey they often aren't: commute times for people who live elsewhere are more than twice as spread out as for residents. When the spreads are similar, the two tests give nearly the same answer; when they differ, Welch's keeps the p-value honest.

In [ ]:
CONFIDENCE = 0.95          # choose 0.90, 0.95, or 0.99 BEFORE looking at results
ALPHA = 1 - CONFIDENCE

def ptext(p):
    return 'p < 0.001' if p < 0.001 else f'p = {p:.3f}'

def sig_words(p):
    return 'statistically significant' if p < ALPHA else 'not statistically significant'

def verdict(p):
    ptext = 'p < 0.001' if p < 0.001 else f'p = {p:.3f}'
    return (f'significant at the {CONFIDENCE:.0%} level ({ptext})' if p < ALPHA
            else f'not significant at the {CONFIDENCE:.0%} level ({ptext})')

def z_test_shares(data, y, x, labels=None):
    '''Z-test of proportions. y: a 1/0 dummy outcome. x: a 1/0 dummy for two groups.'''
    d = data[[y, x]].dropna()
    labels = labels or {0: f'{x} = 0', 1: f'{x} = 1'}
    count = d.groupby(x)[y].sum()
    n = d.groupby(x)[y].count()
    share = count / n
    z, p = proportions_ztest(count[::-1], n[::-1])   # group 1 minus group 0, so the sign matches the difference
    for g in n.index:
        print(f'  {labels[g]:>22}: {share[g]:.1%} (n = {n[g]})')
    gap = (share.iloc[1] - share.iloc[0]) * 100
    print(f'  Difference: {gap:+.1f} percentage points.  Z = {z:.2f}, {verdict(p)}')
    g0, g1 = n.index[0], n.index[1]
    print(f'\n  DRAFT SENTENCE (edit the wording): About {share[g1]:.0%} of {labels[g1]} (n = {n[g1]}) [describe Y], '
          f'compared with {share[g0]:.0%} of {labels[g0]} (n = {n[g0]}), a gap of {abs(gap):.0f} percentage points. '
          f'The difference was {sig_words(p)} at the {CONFIDENCE:.0%} level (Z = {z:.2f}, {ptext(p)}).')
    return share, n, z, p

def t_test_means(data, y, x, labels=None):
    '''Welch t-test of two means. y: a number or 1 to 5 score. x: a 1/0 dummy for two groups.'''
    d = data[[y, x]].dropna()
    labels = labels or {0: f'{x} = 0', 1: f'{x} = 1'}
    g0, g1 = d[d[x] == 0][y], d[d[x] == 1][y]
    t, p = stats.ttest_ind(g1, g0, equal_var=False)
    for g, vals in [(0, g0), (1, g1)]:
        print(f'  {labels[g]:>22}: mean {vals.mean():.2f}, median {vals.median():.2f} (n = {len(vals)})')
    print(f'  Difference in means: {g1.mean() - g0.mean():+.2f}.  t = {t:.2f}, {verdict(p)}')
    print(f'\n  DRAFT SENTENCE (edit the wording): {labels[1]} averaged {g1.mean():.1f} [units of Y] (n = {len(g1)}), '
          f'compared with {g0.mean():.1f} for {labels[0]} (n = {len(g0)}). '
          f'The difference was {sig_words(p)} at the {CONFIDENCE:.0%} level (Welch t = {t:.2f}, {ptext(p)}).')
    return t, p

def chi_square(data, y, x):
    '''Chi-square test for a crosstab of two categorical variables.'''
    d = data[[y, x]].dropna()
    table = pd.crosstab(d[x], d[y])
    chi2, p, dof, expected = chi2_contingency(table, correction=False)
    print((pd.crosstab(d[x], d[y], normalize='index') * 100).round(1).to_string())
    print(f'  Chi-square = {chi2:.2f}, df = {dof}, {verdict(p)}')
    small = (expected < 5).sum()
    if small > 0:
        print(f'  Caution: {small} cell(s) have an expected count under 5. Consider combining categories.')
    return chi2, p, dof

def bar_chart_with_moe(data, y, x, labels, title, ylabel):
    '''Bar chart of a share (if y is 1/0) or a mean (otherwise) for each group, with 95% margins of error.'''
    d = data[[y, x]].dropna()
    grp = d.groupby(x)[y]
    is_share = set(d[y].unique()) <= {0, 1}
    center = grp.mean() * (100 if is_share else 1)
    if is_share:
        p_ = grp.mean()
        moe = 1.96 * np.sqrt(p_ * (1 - p_) / grp.count()) * 100
    else:
        moe = 1.96 * grp.std() / np.sqrt(grp.count())
    names = [f'{labels[g]} (n = {grp.count()[g]})' for g in center.index]
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(names, center.values, yerr=moe.values, capsize=6, color=['#5B7FA8', '#2A9D8F', '#F2B134', '#E85C34'][:len(names)])
    if is_share:
        ax.set_ylim(0, 100)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    for i, (c_, m_) in enumerate(zip(center.values, moe.values)):
        ax.text(i, c_ + m_ + (2 if is_share else 0.02 * center.max()), f'{c_:.0f}%' if is_share else f'{c_:.1f}', ha='center')
    plt.tight_layout()
    plt.show()
    print('For your Data Note: error bars are 95% margins of error, which assume a random sample.')

def who_responded(data, columns):
    '''Counts and shares for a few columns: a "who responded" table for your methods section.'''
    for col in columns:
        t = data[col].value_counts(dropna=False).rename('count').to_frame()
        t['percent'] = (t['count'] / t['count'].sum() * 100).round(1)
        print(f'\n{col}')
        print(t.to_string())

## 3. t-test: do residents have shorter commutes?

**Hypothesis:** people who live in the neighborhood where they work have shorter commutes than people who come in from elsewhere.
- Y: commute minutes (Q9), a number
- X: lives here (Q7), a dummy

Look first, then test.

In [ ]:
df.groupby('lives_here_dv')['commute_num'].describe()

In [ ]:
t_test_means(df, 'commute_num', 'lives_here_dv', labels={0: 'Lives elsewhere', 1: 'Lives here'});

## 4. Z-test of proportions: whose housing costs went up?

**Hypothesis:** renters are more likely than owners to say their own rent or mortgage went up in the past 12 months.
- Y: own housing cost increased (3e), a dummy
- X: rents their home (Q8), a dummy

This is the same test you used for ACS shares in Assignment 1.

In [ ]:
z_test_shares(df, 'p12_own_housing_increased_dv', 'rent_home_dv', labels={0: 'Owners', 1: 'Renters'});

### 4.1 A result that isn't significant

Last year's class found renters more likely than owners to support more housing. With this year's question and respondents:

In [ ]:
z_test_shares(df, 'support_housing_agree_dv', 'rent_home_dv', labels={0: 'Owners', 1: 'Renters'});

### 4.2 Why you choose the confidence level first

Feeling connected to the community, renters vs. owners. Try changing `CONFIDENCE` in Section 2 to 0.99 and rerunning both cells: the numbers don't change, but the verdict does. That's why you choose the level before you look.

In [ ]:
z_test_shares(df, 'connected_agree_dv', 'rent_home_dv', labels={0: 'Owners', 1: 'Renters'});

### 4.3 Dummy or score? The same question two ways

A dummy (agree = 1) throws away the difference between "agree" and "strongly agree." The 1 to 5 score keeps it, so a t-test on the score can sometimes detect a difference the Z-test misses. Here, support for more housing by survey mode:

In [ ]:
z_test_shares(df, 'support_housing_agree_dv', 'self_admin_dv', labels={0: 'Read by interviewer', 1: 'Self-administered'})
print()
t_test_means(df, 'support_housing_score', 'self_admin_dv', labels={0: 'Read by interviewer', 1: 'Self-administered'});

## 5. Chi-square: does agreement differ by city?

A crosstab with more than two groups needs a chi-square test. El Cerrito has only five surveys, so first we test the five cities as they are, then with Richmond and El Cerrito combined.

In [ ]:
chi_square(df, 'changed_a_lot_agree_dv', 'city');

In [ ]:
df['city4'] = df['city'].replace({'Richmond': 'Richmond or El Cerrito', 'El Cerrito': 'Richmond or El Cerrito'})
chi_square(df, 'changed_a_lot_agree_dv', 'city4');

## 6. Is it the mode, or the people?

Survey mode is a covariate: not what we're studying, but something that can shape answers. Agreement that long-time residents can afford to stay, by mode:

In [ ]:
z_test_shares(df, 'afford_to_stay_agree_dv', 'self_admin_dv', labels={0: 'Read by interviewer', 1: 'Self-administered'});

In [ ]:
# Interviewer-read surveys were more common in some cities and languages. Check:
print(pd.crosstab(df['city'], df['mode'], normalize='index').round(2), '\n')
# Does the gap hold among English-language surveys only?
z_test_shares(df[df['language'] == 'English'], 'afford_to_stay_agree_dv', 'self_admin_dv',
              labels={0: 'Read by interviewer', 1: 'Self-administered'});

## 7. An exhibit with margins of error

`bar_chart_with_moe` draws the chart for you. For a share, the 95% margin of error is 1.96 × √(p(1 − p)/n); for a mean, 1.96 × (standard deviation ÷ √n). Put the test in the note under the chart.

In [ ]:
bar_chart_with_moe(df, 'p12_own_housing_increased_dv', 'rent_home_dv',
                   labels={0: 'Owners', 1: 'Renters'},
                   title='Own rent or mortgage increased in the past 12 months',
                   ylabel='Percent who said it increased')

## 8. Your turn: test your own hypotheses, step by step

Do this for each of the two hypotheses on your Lab 12 worksheet. It's the same every time.

**Before you start:** scroll up to Section 1 and check the message. It should say **"Loaded YOUR cleaned Lab 12 file."** If it says it cleaned the raw file, finish Lab 12 and save `survey_cleaned_2026.csv` first, so your tests use your own coding decisions.

**Step 1. Write your hypothesis** as a `#` comment in the test cell, with the direction you expect.

**Step 2. Find your variable names.** Run the Step 2 cell. It lists the dummies (`_dv`), scores (`_score`), and numbers (`_num`) in your data. If a variable you need isn't there, go back to Lab 12, make it, save `survey_cleaned_2026.csv` again, and rerun Section 1 here.

**Step 3. Check your variables** with `value_counts(dropna=False)`. Your X should show only 0, 1, and NaN (or a few categories, for chi-square). Your Y should be 0/1, a 1 to 5 score, or a number.

**Step 4. Pick the test:**

| Your outcome (Y) is... | Your explanatory variable (X) is... | Use |
|---|---|---|
| a 0/1 dummy | a 0/1 dummy | `z_test_shares(df, Y, X, LABELS)` |
| a number or a 1 to 5 score | a 0/1 dummy | `t_test_means(df, Y, X, LABELS)` |
| a dummy or a category | a category with more than two groups | `chi_square(df, Y, X)` |
| a number | a number | correlation (Monday, November 23) |

**Step 5. Fill in the test cell:** replace the example names after `Y =` and `X =` with yours (keep the quote marks), say what 0 and 1 mean in `LABELS`, and use the function from Step 4. Run it.

**Step 6. Make your chart** with `bar_chart_with_moe`: fill in the title and the y-axis label. Right-click the chart to save it.

**Step 7. Fill in the P/NP #11 form:** your hypothesis, how you coded Y and X (from Lab 12), the test, the n's, the shares or means, Z or t, the p-value, whether it's significant at your level, the chart, and the draft sentence edited into your own words.

**Pick your hypotheses before you test, and report what you find, significant or not.** If you test twenty *different* comparisons at 95% confidence (twenty different pairs of variables), about one will look significant by chance alone, even if none of the relationships is real. Rerunning the *same* test on the same data always gives the same answer; the chance comes from which people happened to end up in our sample. The optional cell at the end of this section shows it.

In [ ]:
# Step 2: list the variables you can test
[c for c in df.columns if c.endswith('_dv') or c.endswith('_score') or c.endswith('_num')]

In [ ]:
# Step 3: check your variables. Replace the names with yours.
print(df['rent_home_dv'].value_counts(dropna=False), '\n')
print(df['support_housing_agree_dv'].value_counts(dropna=False))

In [ ]:
# TEST 1
# Step 1, my hypothesis: [write it here, with a direction]
# Step 5: change the next three lines to your variables, then use the matching function.
Y = 'support_housing_agree_dv'           # your outcome
X = 'rent_home_dv'                       # your explanatory variable
LABELS = {0: 'owners', 1: 'renters'}     # what 0 and 1 mean for X

z_test_shares(df, Y, X, LABELS);         # or: t_test_means(df, Y, X, LABELS)   or: chi_square(df, Y, X)

In [ ]:
# Step 6: chart for Test 1. Change the title and the y-axis label.
bar_chart_with_moe(df, Y, X, LABELS,
                   title='Support for building more housing, by home tenure',
                   ylabel='Percent who agree')

In [ ]:
# TEST 2
# Step 1, my hypothesis: [write it here, with a direction]
Y = 'commute_num'
X = 'lives_here_dv'
LABELS = {0: 'people who live elsewhere', 1: 'residents'}

t_test_means(df, Y, X, LABELS);

In [ ]:
# Step 6: chart for Test 2
bar_chart_with_moe(df, Y, X, LABELS,
                   title='Average commute, by whether people live in the neighborhood',
                   ylabel='Minutes')

### Optional: see why twenty comparisons produce a false alarm

This cell makes twenty *fake* explanatory variables by flipping a coin for each respondent, so none of them can truly be related to support for more housing. Then it tests each one. Any "significant" result is pure chance: the coin happened to sort slightly more supporters into one group. With a 95% level, expect about one in twenty (sometimes none, sometimes two or three).

In [ ]:
rng = np.random.default_rng(201)        # a fixed seed, so everyone gets the same coin flips
false_alarms = 0
for i in range(20):
    df['coin_flip'] = rng.integers(0, 2, len(df))     # 0 or 1 at random: unrelated to anything
    d = df[['support_housing_agree_dv', 'coin_flip']].dropna()
    z, p = proportions_ztest(d.groupby('coin_flip')['support_housing_agree_dv'].sum(),
                             d.groupby('coin_flip')['support_housing_agree_dv'].count())
    if p < 0.05:
        false_alarms += 1
print(f'{false_alarms} of 20 coin-flip groupings came out "significant" at the 95% level, by chance alone.')
df = df.drop(columns='coin_flip')

## 9. For your methods section: who responded

Your final memo needs a short table describing who answered the survey. Change the list to the columns you want to describe.

In [ ]:
who_responded(df, ['city', 'role', 'mode', 'language'])